# Notebook 2 – Cross-Platform Fine-Tuning (Windows, macOS, Linux)
This notebook uses the Hugging Face ecosystem so it runs on Windows, macOS (Intel and Apple Silicon), Linux, and Google Colab.

## 1. Install Dependencies

In [ ]:
%pip install -q torch transformers datasets accelerate peft trl bitsandbytes huggingface_hub

## 2. Imports

In [1]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig
from trl import SFTTrainer

/Users/amiteshsinha/Training/ai_labs/2026_4_genAI_Lab/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Select a Model

In [2]:
MODEL = "meta-llama/Llama-3.2-1B-Instruct"


## 4. Load Model

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    device_map="auto",
    torch_dtype="auto"
)

Loading weights: 100%|██████████| 146/146 [00:04<00:00, 31.03it/s]


## 5. Load Dataset

In [7]:
dataset = load_dataset("json", data_files="data/train.jsonl", split="train")
dataset

Generating train split: 3 examples [00:00, 461.54 examples/s]


Dataset({
    features: ['messages'],
    num_rows: 3
})

## 6. Configure LoRA

In [8]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

## 7. Training Arguments

In [9]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
)

## 8. Fine-Tune

In [12]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    peft_config=peft_config,
)

trainer.train()

/Users/amiteshsinha/Training/ai_labs/2026_4_genAI_Lab/.venv/lib/python3.13/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/Users/amiteshsinha/Training/ai_labs/2026_4_genAI_Lab/.venv/lib/python3.13/site-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.
/Users/amiteshsinha/Training/ai_labs/2026_4_genAI_Lab/.venv/lib/python3.13/site-packages/torch/utils/data/datal

Step,Training Loss


TrainOutput(global_step=2, training_loss=4.271940231323242, metrics={'train_runtime': 9.5219, 'train_samples_per_second': 0.315, 'train_steps_per_second': 0.21, 'total_flos': 1959448596480.0, 'train_loss': 4.271940231323242, 'entropy': 1.7454144358634949, 'num_tokens': 333.0, 'mean_token_accuracy': 0.3663206249475479, 'epoch': 1.0})

## 9. Save

In [11]:
trainer.save_model("./fine_tuned_model")

In [14]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel


tokenizer = AutoTokenizer.from_pretrained(MODEL)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    device_map="auto",
    torch_dtype="auto"
)

model = PeftModel.from_pretrained(
    base_model,
    "./fine_tuned_model"
)

Loading weights: 100%|██████████| 146/146 [00:12<00:00, 11.95it/s]


In [15]:
prompt = """
Write a short commercial real estate listing
for a 10,000 SF office building in Austin.
"""

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=300,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


system

Cutting Knowledge Date: December 2023
Today Date: 27 Jun 2026

user

Write a short commercial real estate listing
for a 10,000 SF office building in Austin.assistant

Here's a short commercial real estate listing for a 10,000 SF office building in Austin:

**Stunning 10,000 SF Office Space in the Heart of Austin**

**Location:** Downtown Austin, Texas

**Building Details:**

* 10,000 square feet of luxurious office space
* 2 levels of floor space, with a total of 8 office suites
* Ample parking for over 200 vehicles
* High ceilings, modern finishes, and state-of-the-art amenities
* Direct access to the vibrant 6th Street entertainment district and downtown Austin

**Features:**

* 24/7 security and concierge service
* State-of-the-art fitness center and rooftop lounge
* On-site restaurant and bar, offering stunning views of the city
* Secure building with intercom access and secure parking
* Conveniently located near major highways and public transportation options

**Why Austi